# Assignment 2: Logistic Regression on the Titanic Dataset
**SkillNexis ML & AI Internship, Week 2** (also answers *Practice Q1 and Q2*)

**Dataset:** Kaggle, [Titanic: Machine Learning from Disaster](https://www.kaggle.com/c/titanic/data). Download `train.csv` and save it as `data/titanic_train.csv`.

**Goal:** predict whether a passenger survived (`Survived` = 1) or not (0).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)

RANDOM_STATE = 42
Path("images").mkdir(exist_ok=True)

## 1. Load and explore

In [ ]:
DATA_PATH = Path("data/titanic_train.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download train.csv from Kaggle (Titanic) and save it as data/titanic_train.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
print(df.isna().sum()[lambda s: s > 0])
print("\nSurvival rate overall:", round(df["Survived"].mean(), 3))
print("\nSurvival rate by sex:\n", df.groupby("Sex")["Survived"].mean().round(3))
print("\nSurvival rate by class:\n", df.groupby("Pclass")["Survived"].mean().round(3))

## 2. Prepare features
Used: `Pclass, Sex, Age, SibSp, Parch, Fare, Embarked`.
Dropped: `PassengerId, Name, Ticket` (identifiers) and `Cabin` (mostly missing).
Missing `Age`/`Fare` are filled with the median and missing `Embarked` with the most frequent port.

In [ ]:
numeric_cols = ["Age", "SibSp", "Parch", "Fare"]
categorical_cols = ["Pclass", "Sex", "Embarked"]   # Pclass is a category (1st/2nd/3rd), not a quantity

X = df[numeric_cols + categorical_cols].copy()
X["Pclass"] = X["Pclass"].astype(str)
y = df["Survived"]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
])

# stratify keeps the survived/died ratio the same in train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)

## 3. Train the Logistic Regression model (Practice Q1)

In [ ]:
log_reg = Pipeline([("prep", preprocess),
                    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
log_reg.fit(X_train, y_train)

## 4. Accuracy and confusion matrix (Practice Q2)

In [ ]:
y_pred = log_reg.predict(X_test)

print(f"Test accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Train accuracy: {accuracy_score(y_train, log_reg.predict(X_train)):.4f}")
print(f"Baseline (always predict the majority class): {max(y_test.mean(), 1 - y_test.mean()):.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(cm, display_labels=["Died", "Survived"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Logistic Regression: Confusion Matrix")
plt.tight_layout()
plt.savefig("images/titanic_confusion_matrix.png", dpi=150)
plt.show()

How to read it: rows are the **actual** class, columns the **predicted** class.
Top-left = true negatives, bottom-right = true positives; the other two cells are the mistakes.

In [ ]:
print(classification_report(y_test, y_pred, target_names=["Died", "Survived"]))

## 5. Which features matter?
Positive coefficient = raises the odds of survival; negative = lowers them.

In [ ]:
names = log_reg.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(log_reg.named_steps["model"].coef_[0], index=names).sort_values()
coefs.plot.barh(figsize=(7, 5), color=["tab:red" if c < 0 else "tab:green" for c in coefs])
plt.title("Logistic Regression coefficients")
plt.axvline(0, color="k", linewidth=0.8)
plt.tight_layout()
plt.savefig("images/titanic_coefficients.png", dpi=150)
plt.show()

## 6. Conclusions
Fill these in from *your* output:
* Test accuracy: **...**, compared with the majority-class baseline of **...**
* Biggest driver of survival (see coefficients): **...**
* Which mistake is more common, false positives or false negatives? **...**